In [1]:
import pandas as pd 
import scipy.stats
import pickle 
import os
from config import supp_table_path, llm_review_path, excluded_models

In [2]:
model_to_display_name_a = {
     'carp_38M' : "CARP (38M)",
     'carp_640M': "CARP (640M)",
     'carp_76M': "CARP (76M)",
     'esm_1_43M': "ESM-1 (43M)",
     'esm_1_85M': "ESM-1 (85M)",
     'esm_1_UR100': "ESM-1 (670M) UR100",
     'esm_1_UR50': "ESM-1 (670M) UR50S",
     'esm_1_UR50D': "ESM-1 (670M) UR50D",
     'esm_1b': "ESM-1b",
     'esm_1v': "ESM-1v",
     'esm_2': "ESM-2 (3B)",
     'esm_2_150M': "ESM-2 (150M)",
     'esm_2_15B': "ESM-2 (15B)",
     'esm_2_35M': "ESM-2 (35M)",
     'esm_2_650M': "ESM-2 (650M)",
     'esm_2_8M': "ESM-2 (8M)",
     'progen2_bfd': "Progen2 (2.7B) BFD90",
     'progen2_large': "Progen2 (2.7B) UR90+BFD30",
     'progen2_medium': "Progen2 (764M) UR90+BFD30",
     'protbert': "ProtBERT",
     'prott5_xl': "ProtT5-XL (3B) UR50",
}


model_to_display_name_b = {
    "esm_1_43M_UR50S" : "ESM-1 (43M)",
    "esm_1_85M_UR50S"  : "ESM-1 (43M)",
    'esm_1_UR100': "ESM-1 (670M) UR100",
    'esm_1_UR50S': "ESM-1 (670M) UR50S",
    'esm_1_UR50D': "ESM-1 (670M) UR50D",
    'protbert': "ProtBERT",
    'esm_1b': "ESM-1b",
    'esm_1v': "ESM-1v",
    'esm_2': "ESM-2 (3B)",
    'esm_2_150M': "ESM-2 (150M)",
    'esm_2_15B': "ESM-2 (15B)",
    'esm_2_35M': "ESM-2 (35M)",
    'esm_2_650M': "ESM-2 (650M)",
    'esm_2_8M': "ESM-2 (8M)"
}

In [3]:
corr_df = pd.read_csv(supp_table_path + "revisions/corr_df_dms.csv", index_col = 0)
e1 = list(filter(lambda x : x in excluded_models, corr_df.columns))
corr_df = corr_df.drop(columns = e1)
dms_res_out = []
for index, row in corr_df.iterrows():
    for k, v in row.items():
        dms_res_out.append([index, "DMS", model_to_display_name_a[k],  v])
dms_res_out = pd.DataFrame(dms_res_out, columns = ["Identifier", "Task","Model", "Performance"])


corr_df = pd.read_csv(supp_table_path + "revisions/corr_df_conservation.csv", index_col = 0)
e1 = list(filter(lambda x : x in excluded_models, corr_df.columns))
corr_df = corr_df.drop(columns = e1)
conservation_res_out = []
for index, row in corr_df.iterrows():
    for k, v in row.items():
        conservation_res_out.append([index, "Site Conservation Estimation", model_to_display_name_a[k],  v])
conservation_res_out = pd.DataFrame(conservation_res_out, columns = ["Identifier", "Task","Model", "Performance"])


corr_df = pd.read_csv(supp_table_path + "revisions/corr_df_cat_sites.csv", index_col = 0)
e1 = list(filter(lambda x : x in excluded_models, corr_df.columns))
corr_df = corr_df.drop(columns = e1)
cat_site_res_out = []
for index, row in corr_df.iterrows():
    for k, v in row.items():
        cat_site_res_out.append([index, "Catalytic Site Identification", model_to_display_name_a[k],  v])
        
cat_site_res_out = pd.DataFrame(cat_site_res_out, columns = ["Identifier", "Task","Model", "Performance"])
table_alpha = pd.concat([dms_res_out, conservation_res_out, cat_site_res_out])


identifier_to_entropy = {}

corr_ent_df = pd.read_csv(supp_table_path + "revisions/corr_entropy_df_dms.csv", index_col = 0)
e1 = list(filter(lambda x : x in excluded_models, corr_ent_df.columns))
corr_ent_df = corr_ent_df.drop(columns = e1)
for index, row in corr_ent_df.iterrows():
    for k, v in row.items():
        ident = index + " " + model_to_display_name_a[k]
        identifier_to_entropy[ident] = v
        
        
corr_ent_df = pd.read_csv(supp_table_path + "revisions/corr_entropy_df_cat_sites.csv", index_col = 0)
e1 = list(filter(lambda x : x in excluded_models, corr_ent_df.columns))
corr_ent_df = corr_ent_df.drop(columns = e1)
for index, row in corr_ent_df.iterrows():
    for k, v in row.items():
        ident = index + " " + model_to_display_name_a[k]
        identifier_to_entropy[ident] = v

        
corr_ent_df = pd.read_csv(supp_table_path + "revisions/corr_entropy_df_conservation.csv", index_col = 0)
e1 = list(filter(lambda x : x in excluded_models, corr_ent_df.columns))
corr_ent_df = corr_ent_df.drop(columns = e1)
for index, row in corr_ent_df.iterrows():
    for k, v in row.items():
        ident = index + " " + model_to_display_name_a[k]
        identifier_to_entropy[ident] = v

table_alpha["match_ident"] = table_alpha["Identifier"] + " " + table_alpha["Model"]
table_alpha["Average Site Entropy"] = table_alpha["match_ident"].apply(lambda x : identifier_to_entropy[x])
table_alpha = table_alpha.drop(columns = ["match_ident"])
table_alpha.to_csv(supp_table_path + "revisions/supp_table_alpha.csv", index = False)



In [4]:
# supp table beta, likely table S3
# performance relationships between all model pairs

corr_df = pd.read_csv(supp_table_path + "revisions/corr_df_dms.csv", index_col = 0)
e1 = list(filter(lambda x : x in excluded_models, corr_df.columns))
corr_df = corr_df.drop(columns = e1)


dms_results = []
model_list = corr_df.columns
for m1 in model_list:
    for m2 in model_list:
        if m1 == m2:
            continue 
            
        x1 = corr_df[m1].values
        x2 = corr_df[m2].values
        
        corr, p = scipy.stats.pearsonr(x1, x2)
        
        m_list = [m1, m2]
        sorted_model_string = "_".join(sorted(m_list))
        dms_results.append([m1, m2, "DMS", corr, p, sorted_model_string])

dms_results = pd.DataFrame(dms_results, columns = ["Model 1", "Model 2", "Task", "Pearson Correlation", "Pearson p-value", "mstring"])
dms_results = dms_results.drop_duplicates(subset = "mstring")


corr_df = pd.read_csv(supp_table_path + "revisions/corr_df_cat_sites.csv", index_col = 0)
e1 = list(filter(lambda x : x in excluded_models, corr_df.columns))
corr_df = corr_df.drop(columns = e1)

cat_sites_results = []
model_list = corr_df.columns
for m1 in model_list:
    for m2 in model_list:
        if m1 == m2:
            continue 
            
        x1 = corr_df[m1].values
        x2 = corr_df[m2].values
        
        corr, p = scipy.stats.pearsonr(x1, x2)
        
        m_list = [m1, m2]
        sorted_model_string = "_".join(sorted(m_list))
        cat_sites_results.append([m1, m2, "Catalytic Site Identification", corr, p, sorted_model_string])

cat_sites_results = pd.DataFrame(cat_sites_results, columns = ["Model 1", "Model 2", "Task", "Pearson Correlation", "Pearson p-value", "mstring"])
cat_sites_results = cat_sites_results.drop_duplicates(subset = "mstring")

corr_df = pd.read_csv(supp_table_path + "revisions/corr_df_conservation.csv", index_col = 0)
e1 = list(filter(lambda x : x in excluded_models, corr_df.columns))
corr_df = corr_df.drop(columns = e1)



conservation_results = []
model_list = corr_df.columns
for m1 in model_list:
    for m2 in model_list:
        if m1 == m2:
            continue 
            
        x1 = corr_df[m1].values
        x2 = corr_df[m2].values
        
        corr, p = scipy.stats.pearsonr(x1, x2)
        
        m_list = [m1, m2]
        sorted_model_string = "_".join(sorted(m_list))
        conservation_results.append([m1, m2, "Site Conservation Estimation", corr, p, sorted_model_string])

conservation_results = pd.DataFrame(conservation_results, columns = ["Model 1", "Model 2", "Task", "Pearson Correlation", "Pearson p-value", "mstring"])
conservation_results = conservation_results.drop_duplicates(subset = "mstring")


table_beta = pd.concat([dms_results, conservation_results, cat_sites_results])
table_beta["Model 1"] = table_beta["Model 1"].apply(lambda x : model_to_display_name_a[x])
table_beta["Model 2"] = table_beta["Model 2"].apply(lambda x : model_to_display_name_a[x])
table_beta = table_beta.drop(columns = "mstring")
table_beta.to_csv(supp_table_path + "revisions/supp_table_beta.csv", index = False)


In [5]:
# same as before revisions

# supp table epsilon , skipped delta and gamma since used elsewhere 
# optimal model choices based on entropy for all tasks 

dms_maple_choices = pd.read_csv(supp_table_path + "entropy_optimal_model_choices_DMS.csv", index_col = 0)
cat_site_maple_choices = pd.read_csv(supp_table_path + "entropy_optimal_model_choices_cat_sites.csv")
conservation_model_choices = pd.read_csv(supp_table_path + "entropy_optimal_model_choices_conservation.csv", index_col = 0)

combined = pd.concat([dms_maple_choices, cat_site_maple_choices, conservation_model_choices])
combined.to_csv(supp_table_path + "revisions/supp_table_epsilon.csv")

In [6]:
# supp table zeta
# homologs by our definition for zero shot tasks 


df0 = pd.read_csv(supp_table_path + "revisions/r_hom_counts_dms.csv")
df0 = df0.rename(columns = {"Unnamed: 0" : "Identifier"})
df0["Task"] = "DMS"
df1 = pd.read_csv(supp_table_path + "revisions/r_homologs_all_datasets_cat_sites.csv")
df2 = pd.read_csv(supp_table_path + "revisions/r_homologs_all_datasets_conservation.csv")
   
combined = pd.concat([df0, df1, df2])
combined.to_csv(supp_table_path + "revisions/supp_table_zeta.csv", index = False)

In [7]:
# Same as pre revision
# maple based model selections for three zero shot tasks 
lst = [
    "entropy_optimal_model_choices_cat_sites.csv",
    "entropy_optimal_model_choices_conservation.csv",
    "entropy_optimal_model_choices_DMS.csv"
]

df_pieces = []
for fn in lst: 
    df = pd.read_csv(supp_table_path + fn)
    df_pieces.append(df)
    
combined = pd.concat(df_pieces)
combined = combined.drop(columns = ["Unnamed: 0"])
combined.to_csv(supp_table_path + 'revisions/supp_table_kappa.csv')

In [8]:
# alignment pred results 

dms_aln = pd.read_csv(supp_table_path + "revisions/aln_scores_dms.csv", index_col = 0)
dms_aln["Task"] = "DMS"
cat_site_aln = pd.read_csv(supp_table_path + "revisions/cat_site_aln_scores.csv", index_col = 0)
cat_site_aln = cat_site_aln.rename(columns = {"bfd" : "BFD"})
cat_site_aln["Task"] = "Catalytic Site Identification"
combined = pd.concat([dms_aln, cat_site_aln])
combined.to_csv(supp_table_path + "revisions/supp_table_eta.csv")


In [9]:
!ls ../../../revisions/notebooks/fine_tuned_summary_results/



ppi_affinity.pkl  total_performance_ppi_affinity.pkl
scl.pkl		  total_performance_scl.pkl
solubility.pkl	  total_performance_solubility.pkl


In [10]:
# newly added downstream training 
downstream_path = "../../../revisions/notebooks/downstream_summary_results/"

task_to_metric = {
    "ppi_affinity" : "pearson_correlation",
    "scl" : "f1_macro",
    "solubility" : "auprc"
}
removed_models = ["carp_600k"]
task_to_str = {
    "scl" : "Subcellular Localization", "solubility" : "Solubility", "ppi_affinity" : "PPI Affinity"
}

data_df = [] 

for task in ["scl", "solubility", "ppi_affinity"]:
    with open(downstream_path + task + "_mean_pooled.pkl", "rb") as f1:
        data = pickle.load(f1)
    f1.close()
    
    with open(downstream_path + "total_performance_" +  task + "_mean_pooled.pkl", "rb") as f1:
        overall_data = pickle.load(f1)
    f1.close()
    
    
    
    
    for m in data.keys():
        if m in removed_models:
            continue 
            
            
        if m == "esm_2_3B":
            display_name = "ESM-2 (3B)"
        else:
            display_name = model_to_display_name_a[m]
        
        overall_res =  overall_data[m]
        overall_res = "|".join(list(map(lambda x : str(round(x, 6)), overall_res)))
        
        
        data_df.append([
            task_to_str[task], 
            task_to_metric[task], 
            "mean_pooled", 
            "Downstream NN",  
            display_name, 
            data[m]["upper"], 
            data[m]["lower"],
            overall_res
        ])
        
for task in ["scl", "solubility", "ppi_affinity"]:
    with open(downstream_path + task + "_cls_pooled.pkl", "rb") as f1:
        data = pickle.load(f1)
    f1.close()
    
    with open(downstream_path + "total_performance_" +  task + "_cls_pooled.pkl", "rb") as f1:
        overall_data = pickle.load(f1)
    f1.close()
    
    for m in data.keys():
        if m in removed_models:
            continue 
            
            
        if m == "esm_2_3B":
            display_name = "ESM-2 (3B)"
        else:
            display_name = model_to_display_name_a[m]
        
        overall_res =  overall_data[m]
        overall_res = "|".join(list(map(lambda x : str(round(x, 6)), overall_res)))
        
        data_df.append([
            task_to_str[task], 
            task_to_metric[task], 
            "CLS_pooled", 
            "Downstream NN",  
            display_name, 
            data[m]["upper"], 
            data[m]["lower"],
            overall_res
        ])

        
fine_tune_path = "../../../revisions/notebooks/fine_tuned_summary_results/"

for task in ["scl", "solubility", "ppi_affinity"]:
    with open(fine_tune_path + task + ".pkl", "rb") as f1:
        data = pickle.load(f1)
    f1.close()

    with open(fine_tune_path + "total_performance_" + task + ".pkl", "rb") as f1:
        overall_data = pickle.load(f1)
    f1.close()
        
    for m in data.keys():
        if m in removed_models:
            continue 
            
            
        if m == "esm_2_3B":
            display_name = "ESM-2 (3B)"
        else:
            display_name = model_to_display_name_a[m]
        
        if "top" in  data[m].keys():
            
            upper = data[m]["top"]
            lower = data[m]["bottom"]
            if type(upper) == list:
                upper = "|".join(list(map(lambda x : str(x), upper)))
                lower = "|".join(list(map(lambda x : str(x), lower)))
                
        else:
            upper = data[m]["upper"]
            lower = data[m]["lower"]

        overall_res = overall_data[m]
        overall_res = "|".join(list(map(lambda x : str(round(x, 6)), overall_res)))
        
        data_df.append([
            task_to_str[task], 
            task_to_metric[task], 
            "mean_pooled", 
            "Fine Tuned",  
            display_name, 
            upper, 
            lower,
            overall_res
        ])
        
        
data_df = pd.DataFrame(data_df, columns = [
    "Task" , 
    "Reported Metric", 
    "Pool Method", 
    "Training Setup", 
    "Model", 
    "High Homolog Count Results",
    "Low Homolog Count Results",
    "Overall Results"
])


data_df.to_csv(supp_table_path + "revisions/downstream_results.csv", index = False)


In [11]:
data_df["Model"].value_counts()

Model
ProtBERT                     9
ESM-2 (650M)                 9
Progen2 (764M) UR90+BFD30    9
ProtT5-XL (3B) UR50          6
ESM-2 (15B)                  6
Progen2 (2.7B) UR90+BFD30    6
ESM-1v                       6
CARP (640M)                  6
ESM-1 (85M)                  6
ESM-1 (43M)                  6
ESM-1 (670M) UR50D           6
ESM-1 (670M) UR50S           6
ESM-1b                       6
ESM-2 (8M)                   6
ESM-2 (35M)                  6
ESM-2 (150M)                 6
ESM-2 (3B)                   6
ESM-1 (670M) UR100           6
CARP (38M)                   3
CARP (76M)                   3
Name: count, dtype: int64

In [12]:
data_df

,Task,Reported Metric,Pool Method,Training Setup,Model,High Homolog Count Results,Low Homolog Count Results,Overall Results
0,Subcellular Localization,f1_macro,mean_pooled,Downstream NN,ProtT5-XL (3B) UR50,0.547913|0.552979|0.551493|0.550857|0.548793|0...,0.522343|0.522998|0.523822|0.525017|0.527881|0...,0.55023|0.551841|0.549994|0.551485|0.551119|0....
1,Subcellular Localization,f1_macro,mean_pooled,Downstream NN,ESM-2 (15B),0.715268|0.723062|0.720957|0.721741|0.722167|0...,0.580066|0.569805|0.580711|0.56833|0.580711|0....,0.725814|0.727802|0.727765|0.725771|0.727325|0...
2,Subcellular Localization,f1_macro,mean_pooled,Downstream NN,ESM-2 (3B),0.582371|0.582371|0.581483|0.582371|0.58253|0....,0.526515|0.527494|0.528093|0.525302|0.528104|0...,0.591019|0.585157|0.591177|0.584973|0.583|0.58...
3,Subcellular Localization,f1_macro,mean_pooled,Downstream NN,ESM-2 (650M),0.553304|0.552203|0.556276|0.553171|0.556498|0...,0.495784|0.493244|0.495758|0.490888|0.495166|0...,0.546343|0.546921|0.547093|0.54745|0.548171|0....
4,Subcellular Localization,f1_macro,mean_pooled,Downstream NN,ESM-2 (150M),0.496864|0.500458|0.500194|0.502718|0.499377|0...,0.452508|0.450907|0.450733|0.453878|0.447829|0...,0.485472|0.487181|0.487003|0.486946|0.484934|0...
...,...,...,...,...,...,...,...,...
118,PPI Affinity,pearson_correlation,mean_pooled,Fine Tuned,ProtT5-XL (3B) UR50,0.417317|0.616168|0.599489|0.473723|0.567723|0...,0.470865|0.408724|0.572581|0.379497|0.43185|0....,0.409765|0.529143|0.578872|0.424627|0.492652|0...
119,PPI Affinity,pearson_correlation,mean_pooled,Fine Tuned,ESM-2 (650M),0.380689|0.375432|0.392147|0.37805|0.355092|0....,0.319239|0.321049|0.207199|0.239028|0.333653|0...,0.320742|0.311504|0.295748|0.278021|0.312908|0...
120,PPI Affinity,pearson_correlation,mean_pooled,Fine Tuned,CARP (640M),0.642643|0.621652|0.556462|0.653595|0.583594|0...,0.659586|0.586442|0.616488|0.595001|0.654167|0...,0.652658|0.614204|0.581769|0.637481|0.615546|0...
121,PPI Affinity,pearson_correlation,mean_pooled,Fine Tuned,Progen2 (764M) UR90+BFD30,0.781513|0.602209|0.793579|0.792296|0.744167|0...,0.711177|0.659148|0.728799|0.688983|0.743601|0...,0.741706|0.632251|0.763654|0.745933|0.739261|0...


In [13]:
supp_table_descriptions = {
    "Supplementary Table 1": "Model performance and average site entropy for all models and sequences included in zero-shot analyses.",
    "Supplementary Table 2": "Performance correlations between all model pairs for proteins involved in each of the zero shot predictive tasks.",
    "Supplementary Table 3": "Homolog count for each protein involved in each of the zero shot predictive tasks for all datasets included in analysis.",
    "Supplementary Table 4": "Performance results for supervised training on PPI affinity, subcellular localization, and solubility predictions.",
    "Supplementary Table 5": "Alignment-based zero-shot performance on each sequence involved in deep mutational scanning (DMS) variant effect prediction and catalytic site identification prediction using alignments from the three datasets included in this analysis.",
    "Supplementary Table 6": "DMS experiments and entropy values used in fitting $\gamma^*$ for use in entropy based model selection with MAPLE.",
    "Supplementary Table 7": "Results of MAPLE-based model selection for each sequence involved in zero-shot tasks.",
    "Supplementary Table 8": "ClinVar AUROC performance for all models when applied to the balanced set of B/LP and P/LP ClinVar variants, as well as the ClinVar variants included in the ProteinGym benchmark, and MAPLE selections for each gene in these datasets.",
    "Supplementary Table 9": "Results of MAPLE-based model selection for all proteins in the human proteome.",
    "Supplementary Table 10": "Minimum distance (Angstroms) for long range pairs of amino acid residues  (over 1022 amino acids apart) for structures in the AlphaFold2 human proteome predictions, and experimentally derived PDB entries filtered for similarity using PISCES. Structures included are chains over 1022 amino acids in length.",
    "Supplementary Table 11": "Tiling each protein in the human proteome over 1022 amino acids long using the Brandes et al. approach, the proportion of amino acid positions covered by multiple tiles that receive conflicting predictions for the reference amino acid (highest probability) when masking each position using each PLM.",
    "Supplementary Table 12": "Average entropy of sequence positions 1-1000 within protein groups of varying lengths. Entropy is reported using the untiled rotary positional embedding (RoPE) approach adopted by ESM-2 models, and the entropy resulting from combining results using the tiled approach.",
    "Supplementary Table 13": "ClinVar P/LP, B/LB, and VUS scores from ESM-2 models implementing RoPE. Classification summary statistics from fitting $t_\theta$ on short proteins, and testing classification accuracy on short protein variants, plus long protein variants using the tiled and untiled approach for scoring.",
    "Supplementary Table 14": "Variants included in class-balanced ClinVar analyses of model predictive capabilities among variants in long vs. short proteins, with scores listed using the tiled approach that is applicable to all models.",
    "Supplementary Table 15": "Results of train/test splits fitting $t_\theta$ for model VEP scores on variants among randomly selected 50\% of proteins, and testing $t_\theta$ on the excluded 50\% of proteins. Variants and proteins in this table are separated independently of length.",
    "Supplementary Table 16": "SCOPe selections and cosine similarities representing pairs of proteins within the same family, same superfamily, or same fold.",
    "Supplementary Table 17": "Sequences included from Orphan25, human selections, RandSeq, and Markov-based groups with varying window sizes.",
    "Supplementary Table 18": "Intra-group cosine similarities between sequence embedding pairs in Orphan25 and the human selections for ESM-2 models.",
    "Supplementary Table 19": "Dimensional variance for embeddings of proteins from Orphan25, 25 human selections, bottom 25 proteins by homolog count in SCOPe, and top 25 proteins by homolog count in SCOPe.",
    "Supplementary Table 20": "Cosine similarities between protein pair embeddings designated as the top 100 and bottom 100 pairs by homolog count chosen from different SCOPe folds for ESM-2 models.",
    "Supplementary Table 21": "Intra-group cosine similarities for pairs of proteins within RandSeq, and Markov-based random generations with varying window sizes.",
    "Supplementary Table 22": "Randomly chosen 25,000 proteins from UniRef50, homolog count for each of them, their distance to the orphanage using CLS and mean pooled embeddings, and principal components 1 and 2 obtained from CLS and mean pooled embeddings.",
    "Supplementary Table 23": "Canonical human sequences used for all analyses.",
    "Supplementary Table 24": "Sequence information and labels used for zero-shot catalytic site inference."
}
table_descriptions = [[k,v] for (k,v) in supp_table_descriptions.items()]


table_descriptions = pd.DataFrame(table_descriptions, columns = ["Table Number", "Description"])
table_descriptions

,Table Number,Description
0,Supplementary Table 1,Model performance and average site entropy for...
1,Supplementary Table 2,Performance correlations between all model pai...
2,Supplementary Table 3,Homolog count for each protein involved in eac...
3,Supplementary Table 4,Performance results for supervised training on...
4,Supplementary Table 5,Alignment-based zero-shot performance on each ...
5,Supplementary Table 6,DMS experiments and entropy values used in fit...
6,Supplementary Table 7,Results of MAPLE-based model selection for eac...
7,Supplementary Table 8,ClinVar AUROC performance for all models when ...
8,Supplementary Table 9,Results of MAPLE-based model selection for all...
9,Supplementary Table 10,Minimum distance (Angstroms) for long range pa...


In [14]:
writer = pd.ExcelWriter("r_submission_tables.xlsx", engine = 'xlsxwriter')

######################################################


table_descriptions.to_excel(writer, sheet_name = "Descriptions", index = False)

# fig 2 + 3  

# performance zero shot with entropy 
all_performance = pd.read_csv(supp_table_path + "revisions/supp_table_alpha.csv")
all_performance.to_excel(writer, sheet_name = 'S1', index = False)

#pairwise model correlations 
pairwise_corrs = pd.read_csv(supp_table_path + "revisions/supp_table_beta.csv")
pairwise_corrs.to_excel(writer, sheet_name = 'S2', index = False)

# homologs for tasks 
homolog_counts = pd.read_csv(supp_table_path + "revisions/supp_table_zeta.csv")
homolog_counts.to_excel(writer, sheet_name = 'S3', index = False)



# downstream tasks and fine tuned 
downstream_results =  pd.read_csv(supp_table_path + "revisions/downstream_results.csv")
downstream_results.to_excel(writer, sheet_name = 'S4', index = False)

# aln results
aln_scores = pd.read_csv(supp_table_path + "revisions/supp_table_eta.csv")

aln_scores.index.name = 'Identifier'
aln_scores.to_excel(writer, sheet_name = 'S5', index = False)

######################################################

# Fig 4  

# same as before , MAPLE didnt change 
# maple training 
maple_train_experiments =  pd.read_csv(supp_table_path  + "entropy_optimal_training_experiments.csv")
maple_train_experiments = maple_train_experiments.drop(columns = list(filter(lambda x : "Unnamed" in x, list(maple_train_experiments.columns))))
maple_train_experiments["model"] = maple_train_experiments["model"].apply(lambda x : model_to_display_name_b[x])
maple_train_experiments.to_excel(writer, sheet_name = 'S6', index = False)


# Same as before 
# MAPLE selections and performance for clinvar genome wide 
# indexing is differnt but is correct - index_col = 0, index = False
maple_choices_zero_shot = pd.read_csv(supp_table_path + "revisions/supp_table_kappa.csv", index_col = 0)
maple_choices_zero_shot.to_excel(writer, sheet_name = 'S7', index = False)


# indexing is differnt but is correct - index_col = 0, index = False
clinvar_maple = pd.read_csv(supp_table_path  + "clinvar_auroc_vals_with_MAPLE_annotations.csv", index_col = 0)
clinvar_maple.to_excel(writer, sheet_name = 'S8', index = False)


# indexing is differnt but is correct - index_col = 0, index = False
g_w_maple = pd.read_csv(supp_table_path +  "genome_wide_maple_selections_gamma_1.0.csv")
g_w_maple.to_excel(writer, sheet_name = 'S9', index = False)


######################################################

# Fig 5 

#same 
long_range_entries = pd.read_csv(supp_table_path + "long_range_entries.csv")
long_range_entries.to_excel(writer, sheet_name = 'S10', index = False)

# new 
conflicting_refs = pd.read_csv(supp_table_path + 'revisions/r_gene_ref_aas_different.csv', index_col = 0)
conflicting_refs.to_excel(writer, sheet_name = 'S11', index = False)

# same 
pos_entropy_tiled_untiled = pd.read_csv(supp_table_path + "positional_entropies_1_through_1000_tiled_and_rotary.csv", index_col = 0)
pos_entropy_tiled_untiled.to_excel(writer, sheet_name = 'S12', index = False)

# same
clinvar_a = pd.read_csv(supp_table_path + "clinvar_misclass_summary_stats_with_scores_rotary_tiled.csv", index_col = 0)
clinvar_a.to_excel(writer, sheet_name = 'S13', index = False)

# new 
clinvar_b = pd.read_csv(supp_table_path + "revisions/r_clinvar_misclassification_all_models_all_variants.csv")
clinvar_b = clinvar_b.loc[clinvar_b["model"] != "ProtBERT BFD100"]
clinvar_b.to_excel(writer, sheet_name = 'S14', index = False)

# new - summary stats 
clinvar_c = pd.read_csv(supp_table_path + "revisions/r_long_clinvar_classification_summary_stats.csv")
clinvar_c = clinvar_c.loc[clinvar_c["model"]!="ProtBERT BFD100"]
clinvar_c.to_excel(writer, sheet_name = 'S15', index = False)

######################################################

# Fig 6  

# new with revisions 
family_superfamily_fold_choices = pd.read_csv(supp_table_path + "revisions/r_scope_orphanage_choices_with_distances.csv", index_col = 0)
family_superfamily_fold_choices = family_superfamily_fold_choices.drop(
    columns = ["Unnamed: 0"]
)
family_superfamily_fold_choices.to_excel(writer, sheet_name = 'S16', index = False)


# same as old 
# Sequences included from Orphan25, human selections, RandSeq, and Markov-based groups with varying window sizes
emb_sequences = pd.read_csv(supp_table_path + "embedding_space_used_seqeunces.csv")
# didnt end up using these
emb_sequences = emb_sequences.loc[~emb_sequences["group"].isin(["markov_based_sequences_window_13", "markov_based_sequences_window_14", "markov_based_sequences_window_15", "markov_based_sequences_window_16"])]
emb_sequences["is_orphanage_member"] = emb_sequences["group"].apply(lambda x : x != "Human Selections")
emb_sequences.to_excel(writer, sheet_name = 'S17', index = False)

# new with revisions 
# human vs. orphan25 seqs distances blue and green boxplots 
human_v_orphan25 = pd.read_csv(supp_table_path + "revisions/orphan25_human_cosine_sims_green_blue_boxes.csv")
human_v_orphan25.to_excel(writer, sheet_name = 'S18', index = False)

# new with revisions 
# SCOPe pairs comprising differnt folds 
diff_folds_SCOPE = pd.read_csv(supp_table_path + "revisions/scope_all_distances_pink_purple_boxplots.csv")
diff_folds_SCOPE.to_excel(writer, sheet_name = 'S19', index = False)

# same as old
# Dimensional variance for embeddings of proteins from Orphan25, 25 human selections, bottom 25 proteins by homolog count in SCOPe, and top 25 proteins by homolog count in SCOPe
variance_dimensions = pd.read_csv(supp_table_path + "model_joined_protein_embedding_dimension_variance.csv", index_col = 0)
variance_dimensions.to_excel(writer, sheet_name = 'S20', index = False)

# new with revisoins as old
intra_group_random_seqs = pd.read_csv(supp_table_path + "revisions/r_random_seq_intra_group_mean_pool_dists.csv", index_col = 0)
# dont end up using these
intra_group_random_seqs = intra_group_random_seqs.loc[~intra_group_random_seqs["sequence_group"].isin(["markov_window_13", "markov_window_14", "markov_window_15", "markov_window_16"])]
intra_group_random_seqs.to_excel(writer, sheet_name = 'S21', index = False)

# new with revisions 
random_ur50_25000 = pd.read_csv(supp_table_path + "revisions/all_pool_methods_all_models_PCS_and_orphanage_info.csv")
random_ur50_25000.to_excel(writer, sheet_name = 'S22', index = False)


######################################################
# other stuff 
# human seqeunces used in all analyses
human_sequences = pd.read_csv(llm_review_path + "data/uniprotkb_taxonomy_id_9606_AND_reviewed_2023_10_18.tsv", delimiter = "\t")
human_sequences.to_excel(writer, sheet_name = 'S23', index = False)

# supp table for biolip seqs and labels  
biolip_cleaned = pd.read_csv(llm_review_path + "data/biolip/biolip_cleaned.csv.gz", index_col = 0)
biolip_cleaned.to_excel(writer, sheet_name = 'S24', index = False)


writer.close()

# zip file, also keep uncompressed file 
if os.path.exists("r_submission_tables.xlsx.gz"):
    os.system("rm r_submission_tables.xlsx.gz")

os.system("gzip -k -- r_submission_tables.xlsx")

0